# Example 4A: reviewed Balmer fitting analysis

This notebook shows how Spyctres moves from a reviewed spectrum to a reviewed-analysis stellar-parameter fit.

The central idea is simple: starting from the same spectrum as in Example 3, use the broad wings of Hδ, Hγ, and Hβ to constrain the atmospheric parameters, while keeping model-sensitive line cores and data-quality issues explicit in the provenance.

In Spyctres, **analysis-ready** means the result has explicit wavelength/frame/uncertainty/mask/resolution conventions, reviewed residuals, and stability checks against reasonable scientific alternatives. This notebook runs the compact baseline path; Example 4B follows up with bounded sensitivity checks on the same spectrum.

This spectrum is (intentionally) a complex case. For a cleaner first-success spectrum, use Example 1 or the Gaia benchmark validation tools; for reviewed-analysis reasoning, difficult spectra are useful because they make the blockers visible.

## What this example teaches

- how a maintained recipe prepares a transparent Balmer-wing analysis case;
- how sideband normalization, core masking, reader masks, and fit-time continuum are recorded;
- how to separate pre-fit readiness, fit completion, and post-fit interpretation status.

## Requirements

The review path uses bundled data only. Set `RUN_LEVEL` to `"fit"` or `"line_consistency"` only when PHOENIX is configured and you want to run the model fit.

## Expected outputs

A Balmer-case summary, preparation plots, a readiness audit, and—if fitting is enabled—one joint Balmer result plus residual diagnostics.

Note: A reviewed setup makes the assumptions auditable; it does not certify the fitted parameters until residual and stability checks are also reviewed.


## What this example does

Examples 1–3 introduced reading spectra, diagnostic windows, local line fits, masks, and reviewed `FitSetup` objects. This example adds a higher-level concept: a maintained recipe.

A recipe packages a tested preparation pattern and then exposes what it did through summaries, plots, and serialized provenance. Here, the recipe we will use prepares an X-SHOOTER UVB Balmer-wing case.

We will:

1. read and inspect the X-SHOOTER UVB product;
2. use a built-in Balmer Spyctres recipe to prepare Hδ/Hγ/Hβ windows;
3. inspect the recipe summary and preparation plot;
4. build a reviewed-analysis intent setup;
5. optionally run one joint PHOENIX fit;
6. optionally check whether individual Balmer lines give compatible answers.

## 0. Analysis controls

The variables below are just controls for this worked example. (They are not global Spyctres defaults.)

`RUN_LEVEL` keeps the notebook compact:

- `"review"`: read, inspect, prepare the Balmer case, and audit the setup; no PHOENIX fit;
- `"fit"`: also run one joint Balmer PHOENIX fit;
- `"line_consistency"`: also fit the prepared Balmer lines individually as a consistency check.

The baseline uses sideband-normalized windows plus a modest residual multiplicative continuum. We use degree 1 here because a high-degree continuum, after sideband normalization, can start absorbing broad Balmer-wing information. More flexible continuum choices belong in the systematics follow-up.

In [ ]:
import Spyctres as sp

# Use None to let Spyctres find PHOENIX from its normal configuration.
PHOENIX_DIR = None

# Choose one of: "review", "fit", or "line_consistency".
RUN_LEVEL = "fit"
SHOW_PROGRESS = True

# Baseline scientific choices for this worked example.
# These are notebook controls, not hidden global Spyctres defaults.
core_mask_halfwidth_A = 8.0  # exclude ±8 Å around each Balmer-line centre
continuum_degree = 1        # residual multiplicative Legendre continuum degree

run_level = RUN_LEVEL.strip().lower()
if run_level not in {"review", "fit", "line_consistency"}:
    raise ValueError('RUN_LEVEL must be "review", "fit", or "line_consistency".')


## 1. Load the bundled X-SHOOTER UVB spectrum

The reader name describes the **data product**, not only the instrument. Here `xshooter_merge1d` tells Spyctres to read a merged 1-D X-SHOOTER FITS image product and to preserve the wavelength, frame, uncertainty, mask, and resolution metadata it can infer from the file.

You can discover supported readers with:

```python
sp.list_readers()
```

If Spyctres does not yet have a reader for your format, convert your spectrum to a simple calibrated table first, or add a small reader that returns a `SpectrumSegment` with wavelength, flux, optional uncertainty, metadata, and `valid_mask=True` meaning usable.

In [ ]:
spectrum_path = sp.example_data_path("TOO_Gaia21ccu_SCI_SLIT_FLUX_MERGE1D_UVB.fits")
reader = "xshooter_merge1d"

spec = sp.read_spectrum(spectrum_path, reader=reader)

print(spec.summary())
print(spec.provenance_summary())

## 2. First-look audit plot

The audit plot is a generic way to inspect any loaded spectrum before fitting. It shows the raw flux, a local display normalization, and the current usable-pixel mask. The display normalization is not applied to the data; it is only there to make structure easier to see.

In [ ]:
sp.plot_spectrum_audit(
    spec,
    title="Example 4A: generic audit view before fitting",
    figsize=(14.0, 8.0),
)


## 3. Candidate diagnostic windows

Spyctres can rank diagnostic regions that overlap a spectrum. For this hot/blue UVB spectrum, Balmer lines are the main temperature/gravity diagnostics. Ca H/K, Mg, DIBs, and other features are still useful warning or sanity-check regions, but they are not the main atmospheric-parameter constraints in this worked fit.

In [ ]:
# Ask Spyctres to suggest up to six useful diagnostic regions
# that fall within the wavelength coverage of this spectrum.
windows = sp.select_diagnostic_windows(spec, max_windows=6)

# Print a short summary of the selected regions.
print(windows.summary_text(max_rows=6))

# Build the mask used during review:
# - exclude known archive or detector artifacts;
# - mark telluric regions as warnings, but do not mask them automatically;
# - do not include diffuse interstellar-band regions.
reviewed_mask = sp.build_mask(
    spec,
    archive="mask",
    tellurics="warn",
    dibs=False,
)

# Print what was masked and which regions were marked only as warnings.
print(reviewed_mask.summary_text())

# Plot the full spectrum together with the suggested diagnostic windows
# and any regions that require caution during interpretation.
sp.plot_spectrum_audit(
    spec,
    diagnostic_selection=windows,
    warning_regions=reviewed_mask.warning_regions,
    title="Example 4A: audit view with candidate windows and warning regions",
    figsize=(14.0, 8.0),
)


## 4. Prepare a tested Balmer-wing case

Next, we will use the built-in maintained X-SHOOTER Balmer recipe. This notebook will go over the scientific choices and inspect the output, but we will not show explicitly how to do wavelength conversion, segment construction, sideband normalization, and mask-callable bookkeeping.

What this recipe does:
- crop Hδ, Hγ, and Hβ windows;
- attach line-centre metadata in the data wavelength medium;
- sideband-normalize each window;
- exclude the narrow Balmer cores;
- preserve the reader-derived X-SHOOTER UVB resolution metadata;
- expose all of that through `summary_text()`, `plot_preparation()`, and `to_dict()`.

If you want to see the full argument list, run:

```python
help(sp.recipes.prepare_xshooter_balmer_case)
```

For your own recurring workflow, put project-specific wrappers in your own analysis package, for example `my_project/recipes.py`, rather than editing the installed Spyctres package directly.

In [ ]:
# Prepare separate fitting segments around the Balmer lines.
balmer_case = sp.recipes.prepare_xshooter_balmer_case(
    spec,
    window_mode="notebook",           # Use the broader Balmer windows defined for this example.
    norm_mode="sideband",             # Estimate local continuum from regions beside each line.
    sideband_width=10.0,              # Use the recipe sideband definitions around each line.
    sideband_order=1,                 # Fit a straight line through the sidebands.
    core_mask=core_mask_halfwidth_A,  # Exclude the central part of each Balmer line.
)

# Print a summary of the prepared line regions, normalization, masks, and metadata.
print(balmer_case.summary_text())

# Plot each prepared Balmer region so we can inspect the normalization,
# retained wings, reader-rejected pixels, and masked line core before fitting.
balmer_case.plot_preparation(
    title="Example 4A: what the Balmer recipe prepared",
    ncols=2,
    figsize_per_panel=(7.2, 3.6),
)


### Reading the preparation plot

Orange spans mark the continuum sidebands used for local normalization. Pale red spans mark the Balmer cores excluded from the fit. Red `x` markers are pixels not used after the reader mask and recipe masks. The dashed vertical markers show the Balmer line centres in the spectrum's wavelength medium.

This is the key compromise: the fit uses line-wing information, while the harder-to-model cores remain visible but are not allowed to dominate χ². Pixels rejected by the X-SHOOTER quality array are already excluded by the reader; we should not add duplicate masks for those same pixels.

## 5. Build and audit a reviewed-analysis intent setup

The setup below uses the prepared Balmer collection, the recipe's fit regions and exclusion masks, and a modest residual continuum degree. The prepared segments carry the X-SHOOTER UVB resolution metadata from the reader.

The readiness audit answers whether Spyctres can interpret the result for the requested intent. Importantly, reader-rejected pixels are not automatically "unhandled artifacts". They count as unavailable data, but they should not force us to add another mask that duplicates the instrument quality mask.

If visual inspection finds genuinely new bad regions that were not already rejected, define them explicitly in `additional_artifact_regions` below. Leave that list empty when the reader quality mask has already handled the problem.

In [ ]:
# Additional manual artifact regions are optional and start empty.
# Add intervals here only after visual inspection identifies a real problem
# that was not already rejected by the reader's input quality mask.
additional_artifact_regions = [
    # (wavelength_min_A, wavelength_max_A),
    # Example: (4368.0, 4370.0),
]

manual_artifact_masks = ()

if additional_artifact_regions:
    manual_artifact_masks = (
        sp.wavelength_region_exclusion_mask(
            "manually_reviewed_artifacts",
            additional_artifact_regions,
            metadata={
                "reason": "Regions identified during visual inspection",
                "action": "masked",
            },
        ),
    )


In [ ]:
# Combine recipe masks with any genuinely new manual masks.
all_exclusion_masks = balmer_case.combined_exclusion_masks(manual_artifact_masks)
fit_valid_masks = balmer_case.valid_masks_for(manual_artifact_masks)

# Build a standard fitting setup for reviewed-analysis interpretation.
# The continuum degree is stated explicitly so it appears in the setup summary.
setup = balmer_case.suggest_fit_setup(
    mode="standard",
    intent="reviewed_analysis",
    continuum_degree=continuum_degree,
    extra_exclusion_masks=manual_artifact_masks,
)

In [ ]:
# Check whether the prepared Balmer-line data meet the stricter
# requirements for reviewed-analysis interpretation.
reviewed_analysis_audit = sp.analysis_readiness_audit(
    balmer_case.collection,
    regions=balmer_case.fit_regions_by_segment,  # Audit each prepared segment only inside its own fit window.
    exclude_masks=all_exclusion_masks,     # Apply core masks plus any reviewed additions.
)

# Print the complete setup in a readable form.
# The reproducibility hash is omitted here to keep the notebook output concise.
print(setup.summary_text(include_hash=False))

# Report whether any issues still prevent reviewed-analysis interpretation.
print("\nReviewed-analysis ready:", reviewed_analysis_audit["analysis_ready"])
print("Reviewed-analysis blockers:", reviewed_analysis_audit["blockers"])
print("Reviewed-analysis warnings:", reviewed_analysis_audit["warnings"])

In [ ]:
# Let's print and inspect segment-level readiness details.
# The key quantity is "unhandled_artifact_fraction_inside_fit_window":
# pixels already rejected by the reader mask are reported separately and should
# not be masked a second time.
for segment_audit in reviewed_analysis_audit["audit"]["segments"]:
    metrics = segment_audit["artifact_metrics"]
    print(f"\n{segment_audit['name']}")
    print("flags:", segment_audit["interpretation_flags"])
    print(
        "unhandled artifact fraction:",
        f"{metrics['unhandled_artifact_fraction_inside_fit_window']:.2%}",
    )
    print(
        "already rejected by reader/input mask:",
        f"{metrics['already_rejected_input_mask_fraction_inside_fit_window']:.2%}",
    )
    print("flat zero blocks still usable:", metrics["flat_zero_block_count"])

print("\nRecommended response:")
if manual_artifact_masks:
    print("  - Extra user-reviewed artifact masks are active; rerun the fit and inspect residuals.")
elif "artifact_review_required" in reviewed_analysis_audit["blockers"]:
    print("  - Inspect the plotted red x pixels and residuals before adding any new mask.")
    print("  - If they are already reader-quality rejections, do not duplicate them.")
    print("  - If Hδ remains unstable, compare the joint fit with and without Hδ.")
else:
    print("  - No unhandled artifact blocker remains; proceed to exploratory fitting/residual review.")

For this bundled X-SHOOTER UVB product, the relevant bad pixels are already excluded by the reader using the quality array. The main thing to inspect is Hδ, because it has a substantial reader-rejected interval on its red side. Do **not** add a second mask for those same pixels merely to remove a warning.

The workflow is:

1. inspect the preparation plot and segment-level audit;
2. leave `additional_artifact_regions` empty unless visual inspection finds a new bad interval;
3. run the exploratory joint fit with the existing reader and recipe masks;
4. check stability with and without Hδ before treating the Balmer solution as robust.

The audit now separates already-rejected pixels from unhandled artifacts. If `artifact_review_required` remains, it should point to a real unmasked data problem rather than to support padding or reader-quality rejections.

## 6. Optionally run one joint Balmer PHOENIX fit

Set `RUN_LEVEL = "fit"` or `RUN_LEVEL = "line_consistency"` in the control cell to run the PHOENIX fit. Hδ, Hγ, and Hβ are fitted together through `fit_stellar_spectrum()`. They are not three separate Gaussian line fits here.

If the setup is not analysis-ready, the fit can still be run under an explicit exploratory override. That means Spyctres computes the model comparison so we can inspect residuals and plan follow-up checks, but the result is labelled as diagnostic rather than final analysis.

In [ ]:
# No fit result exists until the fitting stage is actually run.
analysis_result = None

# Use the reviewed setup by default.
run_setup = setup

# "fit" runs the baseline joint fit.
# "line_consistency" runs the same baseline fit before the extra checks later.
if run_level in {"fit", "line_consistency"}:

    # A blocked readiness check does not prevent an exploratory fit,
    # but the override records that the result is not final.
    if setup.summary()["ready_for_intent"] is not True:
        run_setup = setup.allow_exploratory(
            reason=(
                "Example 4 tutorial baseline for residual and line-consistency "
                "review; not treated as final analysis."
            )
        )

    # Fit Hδ, Hγ, and Hβ together with the PHOENIX model grid.
    analysis_result = sp.fit_stellar_spectrum(
        balmer_case.collection,
        model="phoenix",
        setup=run_setup,

        # Apply the reviewed masks, including the excluded Balmer cores.
        valid_mask=fit_valid_masks,

        # Location of the locally installed PHOENIX model library.
        phoenix_dir=PHOENIX_DIR,

        # Optionally print elapsed time and progress updates during the fit.
        progress_callback=(
            (lambda event: print(f"[{event.elapsed_s:6.1f}s] {event}", flush=True))
            if SHOW_PROGRESS
            else None
        ),
    )

    # Print the fitted parameters, provenance, flags, and fit-quality diagnostics.
    print(analysis_result.summary_text(include_hash=False, max_flags=8))
    print("\n" + analysis_result.quality_report_text())

else:
    # In review mode, stop after preparing and auditing the spectrum.
    print('RUN_LEVEL is "review", so no PHOENIX fit was run.')

## 7. Inspect residuals

Reviewed analysis should never stop at the parameter table. The plots below show what the joint fit did and where it failed. In the line-window plot, the model can be displayed across masked pixels as a dashed diagnostic curve; those pixels are not included in χ².

In [ ]:
# Only create the diagnostic plots after a fit has been completed.
if analysis_result is not None:

    # Show the main referee-style overview of the fit:
    # observed spectrum, best-fitting model, residuals, and fit diagnostics.
    sp.plot_fit_referee(
        analysis_result,
        layout="stacked",
        flux_ylim_mode="visible",  # Scale the flux axis to the displayed data.
    )

    # Show the same joint fit separately around Hδ, Hγ, and Hβ.
    # The residual panels use pulls:
    #     pull = (observed flux - model flux) / uncertainty
    # Values much larger than ±1 indicate discrepancies beyond the quoted errors.
    sp.plot_model_line_windows(
        analysis_result,
        windows=balmer_case.fit_windows,
        title="Example 4A: joint Balmer fit, shown line by line",
        show_residuals=True,
        residual_kind="pull",
        ncols=2,
        figsize_per_panel=(7.2, 5.2),
    )

else:
    # Review mode prepares and audits the data but does not generate a model.
    print(
        'Set RUN_LEVEL = "fit" or "line_consistency" '
        "to generate the model and residual plots."
    )
    

## 8. Optional individual-line consistency check

The joint fit is efficient, but it can hide line-specific problems. With `RUN_LEVEL = "line_consistency"`, Spyctres fits each prepared Balmer window separately using the same public fitting path. Agreement among the individual-line solutions increases confidence; large shifts point to continuum, mask, LSF, or model-physics systematics.

This is still not a full uncertainty budget. It is the first compact check before moving to the heavier systematics workflow.

In [ ]:
# No individual-line results exist until the consistency check is run.
line_results = None

# This cell follows the RUN_LEVEL chosen in the control cell.
# Set RUN_LEVEL = "line_consistency" there if you want this section to run.

# Run this section only when the more detailed consistency check was requested
# and the joint Balmer fit completed successfully.
if run_level == "line_consistency" and analysis_result is not None:

    # Refit Hδ, Hγ, and Hβ separately using the same PHOENIX setup.
    # This tests whether the individual lines favour compatible parameters
    # or whether one line is driving the joint solution.
    line_results = sp.fit_case_lines_individually(
        balmer_case,
        base_setup=run_setup,

        # Apply any additional artifact masks defined after visual inspection.
        extra_exclusion_masks=manual_artifact_masks,

        model="phoenix",
        phoenix_dir=PHOENIX_DIR,

        # Optionally show elapsed time and progress for each fit.
        progress_callback=(
            (lambda event: print(f"[{event.elapsed_s:6.1f}s] {event}", flush=True))
            if SHOW_PROGRESS
            else None
        ),
    )

    # Print a short summary for each independently fitted Balmer line.
    for label, result in line_results.items():
        print("\n" + label)
        print(result.summary_text(include_hash=False, max_flags=5))

    # Place the separate fits in one comparison table.
    # Large differences between lines indicate sensitivity to masking,
    # data quality, or model limitations.
    line_comparison = sp.compare_fits(
        list(line_results.values()),
        labels=list(line_results.keys()),
    )

    print("\nLine-consistency comparison:")
    print(sp.format_fit_comparison_table(line_comparison))
    print(
        "\nHow to read this comparison: large shifts between individual "
        "Balmer-line fits mean the joint result is sensitive to line choice "
        "and needs the Example 4B stability checks before interpretation."
    )

    # Overlay the joint fit and the individual-line fits in the same panels.
    # Each individual-line model appears only in the line region that fitted it.
    # Dashed model spans show masked pixels for context, not fitted constraints.
    sp.plot_fit_comparison_line_windows(
        [analysis_result, *line_results.values()],
        labels=["joint Balmer fit", *line_results.keys()],
        windows=balmer_case.fit_windows,
        title="Example 4A: joint fit compared with individual-line fits",
        ncols=2,
        figsize_per_panel=(7.2, 3.8),
        footer=(
            "Solid model traces use fitted pixels. Dashed model spans "
            "show masked pixels for context only."
        ),
    )

elif run_level == "line_consistency":
    # The individual-line fits reuse the reviewed setup from the joint fit.
    print(
        "Run the joint fit first; individual-line checks use "
        "the same PHOENIX setup."
    )

else:
    # The quicker "fit" level stops after the joint Balmer fit.
    print(
        'Set RUN_LEVEL = "line_consistency" '
        "to run the individual-line check."
    )

## 9. What have we learned, and what is still missing?

This notebook gives the compact baseline: recipe preparation, setup audit, one joint Balmer fit, residual plots, and an optional individual-line check.

Before claiming reviewed-analysis stellar parameters, run bounded stability checks. Example 4B continues with **the same spectrum** and compares:

- sideband-normalized windows with different residual continuum degrees;
- several Balmer-core mask widths, while reporting information loss;
- modest resolution/LSF perturbations around the header-derived value;
- leave-one-line-out fits.

At this point we have an auditable **candidate** result, not a final calibrated result. The residuals, quality flags, and individual-line agreement tell us whether the baseline is worth taking to the next stage.

The next stage is to decide whether the physical interpretation is stable under reasonable assumptions. We are not only looking for the variant that gives the smallest χ².

For a broader multi-arm consistency check across UVB/VIS/NIR, continue to Example 6 after Example 4B.
